# Nearest-Centroid Classifier From Scratch

**Topic:** Vectors and Matrix Operations &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X_train` `(2000, 8)`, `y_train` `(2000,)`: 8-dimensional points from 5 classes (labels 0–4), each class a Gaussian cloud with its own center.
- `X_test` `(600, 8)`: unlabeled test points. `y_test` is used only by the evaluation cell.
- Clouds overlap a little, so 100% accuracy is not attainable. That is normal.

## Your task

The rule below, one center per class and the label of the nearest center, is the assignment step of k-means in miniature, with the centers fixed instead of updated.

Using **NumPy only** (no scikit-learn):

1. Compute the centroid (mean vector) of each class from the training data.
2. For every test point, compute its Euclidean distance to all 5 centroids (try to do it with broadcasting, not nested Python loops).
3. Set `predictions` to the array of predicted class labels for `X_test`.

The grader checks accuracy only. Use the nearest-centroid method the task describes, that is what this exercise marks.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

rng = np.random.default_rng(21)

n_classes, dim = 5, 8
centers = rng.normal(0, 3.2, (n_classes, dim))

def sample(n_per, seed):
    r = np.random.default_rng(seed)
    X = np.vstack([centers[c] + r.normal(0, 1.6, (n_per, dim)) for c in range(n_classes)])
    y = np.repeat(np.arange(n_classes), n_per)
    idx = r.permutation(len(y))
    return X[idx], y[idx]

X_train, y_train = sample(400, 100)
X_test, _ = sample(120, 200)
print(f"X_train {X_train.shape}, X_test {X_test.shape}, classes {np.unique(y_train)}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

# Regenerated here, never bound in the cell you write in.
_, y_test = sample(120, 200)

TARGET_ACCURACY = 0.95

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

Compute one centroid per class with a masked mean, then use broadcasting to get the (600, 5) matrix of test-point-to-centroid distances in one line. `argmin` over the centroid axis gives the predicted class.

```python
import numpy as np

# One centroid per class: (5, 8)
centroids = np.vstack([X_train[y_train == c].mean(axis=0) for c in range(5)])

# Broadcasting: (600, 1, 8) - (1, 5, 8) -> (600, 5) distances
dists = np.linalg.norm(X_test[:, None, :] - centroids[None, :, :], axis=2)
predictions = dists.argmin(axis=1)
```
